# Exploration and Discovery

The Exploration and Discovery pattern implements multi-agent scientific research workflows. Specialized agents handle distinct phases — literature synthesis, plan formulation, experimentation, and peer review — mirroring academic research processes. A tripartite review mechanism (three reviewers with different evaluation lenses) provides robust quality assessment.

## Implementation with Flyte v2

This notebook reimplements the Agent Laboratory pattern from Chapter 21 (`ReviewersAgent`, `ProfessorAgent`, `PostdocAgent`) using **Flyte v2 primitives + Anthropic API** — replacing OpenAI.

#### Original (OpenAI) vs Flyte v2 (Anthropic) — Key Differences

| Aspect | Original (OpenAI) | Flyte v2 |
|--------|-------------------|----------|
| **Agent classes** | `ProfessorAgent`, `PostdocAgent`, `ReviewersAgent` Python classes | `@env.task` functions — no class boilerplate |
| **LLM client** | `OpenAI(api_key=openai_api_key)` at init time | `AsyncAnthropic` created inside each task |
| **Parallel reviews** | Sequential `get_score()` calls | `asyncio.gather` — 3 reviewers run concurrently |
| **State passing** | Instance variables (`self.plan`, `self.report`, `self.lit_review_sum`) | Typed dataclasses — serializable, inspectable in UI |
| **Review schema** | JSON embedded in a markdown code block | Pydantic `ReviewScore` — validated, typed |
| **Progress** | `print()` statements | `flyte.report` HTML tab updated per phase |
| **Secrets** | Constructor `openai_api_key` param | `flyte.Secret` injected at task runtime |

> **🧭 When to use this pattern — and how Flyte helps**
>
> Use exploration and discovery in large, loosely-defined problem spaces where the goal is to uncover new information, strategies, or solutions rather than execute a known plan. Flyte supports the search loop with parallel fan-out, caching of evaluated candidates, and — when the agent generates code to try — isolated sandboxes for safe execution.

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' anthropic pydantic

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

### 3. Import dependencies and configure the TaskEnvironment

In [1]:

import os
from dataclasses import dataclass, field
from datetime import timedelta
from typing import List

from anthropic import AsyncAnthropic
import flyte
import flyte.report
from pydantic import BaseModel, Field

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="discovery-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.25.0", "pydantic>=2.0.0")
)

discovery_env = flyte.TaskEnvironment(
    name="discovery_agent",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define data models

The original used Python class instance variables (`self.plan`, `self.report`, `self.lit_review_sum`) to pass state between agent phases. In Flyte v2, typed **Pydantic models** replace these — each phase receives its inputs and returns a serializable result visible in the UI. They're Pydantic (not `@dataclass`) because the models nest one another, and Flyte's dataclass serializer can't pack a nested Pydantic model — keeping the whole graph as `BaseModel`s routes it through Flyte's Pydantic transformer.

In [2]:
class LiteratureReview(BaseModel):
    """Output of the literature synthesis phase."""
    topic: str
    key_findings: List[str]
    research_gaps: List[str]
    summary: str


class ResearchPlan(BaseModel):
    """Output of the PostdocAgent plan formulation phase."""
    hypothesis: str
    methodology: str
    expected_outcomes: List[str]
    evaluation_criteria: List[str]


class ResearchReport(BaseModel):
    """Synthetic research report generated by the PostdocAgent."""
    title: str = ""
    abstract: str = ""
    methodology_description: str = ""
    results: str = ""
    conclusion: str = ""


class ReviewScore(BaseModel):
    """
    Structured review output.
    Replaces get_score()'s JSON embedded in a markdown code block.
    """
    summary: str
    strengths: List[str]
    weaknesses: List[str]
    originality: int = Field(ge=1, le=4, description="1=low 2=medium 3=high 4=very high")
    quality: int = Field(ge=1, le=4)
    clarity: int = Field(ge=1, le=4)
    significance: int = Field(ge=1, le=4)
    soundness: int = Field(ge=1, le=4)
    overall: int = Field(ge=1, le=10, description="1=strong reject 10=award quality")
    decision: str = Field(description="Accept or Reject")
    ethical_concerns: bool


class ReviewPanel(BaseModel):
    """Output of ReviewersAgent — three concurrent reviews."""
    reviewer_1: ReviewScore  # harsh but fair — expects strong experiments
    reviewer_2: ReviewScore  # critical — looks for field impact
    reviewer_3: ReviewScore  # open-minded — looks for novelty
    consensus_score: float
    final_decision: str


class ResearchResult(BaseModel):
    """Complete research pipeline output."""
    topic: str
    literature_review: LiteratureReview
    research_plan: ResearchPlan
    report: ResearchReport
    review_panel: ReviewPanel
    readme: str

### 5. Define the specialist agent tasks

The original used class inheritance (`BaseAgent` → `ProfessorAgent`, `PostdocAgent`). In Flyte v2, each role is a plain `@env.task` — no class hierarchy needed. The phase-specific prompts are preserved from the original.

In [ ]:
async def _llm(
    user: str,
    system: str = "You are a rigorous research assistant. Follow the requested output format exactly.",
    max_tokens: int = 1024,
) -> str:
    """Shared async LLM call helper."""
    # max_retries adds SDK backoff/retries for transient server-side blips
    # (HTTP 500 'api_error: Internal server error'), which are out of our control and
    # usually clear on retry. Default is 2; 5 gives a long multi-call pipeline headroom.
    client = AsyncAnthropic(max_retries=5)
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=max_tokens,
        system=system,
        messages=[{"role": "user", "content": user}],
    )
    return response.content[0].text.strip()


@flyte.trace
async def _literature_agent(topic: str) -> LiteratureReview:
    """
    Synthesize literature for a research topic.
    Replaces Agent Laboratory's literature review phase.
    """
    text = await _llm(
        user=f"Conduct a literature review on: {topic}\n\n"
             "Format:\nKEY FINDINGS:\n- ...\n\nRESEARCH GAPS:\n- ...\n\nSUMMARY:\n...",
        max_tokens=1024,
    )
    findings, gaps, summary = [], [], ""
    section = None
    for line in text.splitlines():
        if "KEY FINDINGS" in line:
            section = "findings"
        elif "RESEARCH GAPS" in line:
            section = "gaps"
        elif "SUMMARY" in line:
            section = "summary"
        elif line.strip().startswith("-") and section == "findings":
            findings.append(line.strip()[1:].strip())
        elif line.strip().startswith("-") and section == "gaps":
            gaps.append(line.strip()[1:].strip())
        elif section == "summary" and line.strip():
            summary += line.strip() + " "
    return LiteratureReview(topic=topic, key_findings=findings, research_gaps=gaps, summary=summary.strip())


@flyte.trace
async def _postdoc_plan(lit_review: LiteratureReview) -> ResearchPlan:
    """
    Formulate a research plan based on the literature review.
    Replaces PostdocAgent's 'plan formulation' phase.
    """
    gaps_str = "\n".join(f"- {g}" for g in lit_review.research_gaps[:3])
    text = await _llm(
        user=f"Topic: {lit_review.topic}\nKey gaps:\n{gaps_str}\n\n"
             "Format:\nHYPOTHESIS:\n...\n\nMETHODOLOGY:\n...\n\nEXPECTED OUTCOMES:\n- ...\n\nEVALUATION CRITERIA:\n- ...",
        max_tokens=768,
    )
    hyp, method, outcomes, criteria = "", "", [], []
    section = None
    for line in text.splitlines():
        if "HYPOTHESIS" in line:
            section = "hyp"
        elif "METHODOLOGY" in line:
            section = "method"
        elif "EXPECTED OUTCOMES" in line:
            section = "outcomes"
        elif "EVALUATION CRITERIA" in line:
            section = "criteria"
        elif section == "hyp" and line.strip() and not line.startswith(" ") and ":" not in line[:15]:
            hyp += line.strip() + " "
        elif section == "method" and line.strip() and not line.startswith(" ") and ":" not in line[:15]:
            method += line.strip() + " "
        elif line.strip().startswith("-") and section == "outcomes":
            outcomes.append(line.strip()[1:].strip())
        elif line.strip().startswith("-") and section == "criteria":
            criteria.append(line.strip()[1:].strip())
    return ResearchPlan(
        hypothesis=hyp.strip() or "Novel approach to address identified gap.",
        methodology=method.strip() or "Empirical evaluation with controlled experiments.",
        expected_outcomes=outcomes or ["Improved performance on benchmarks"],
        evaluation_criteria=criteria or ["Statistical significance", "Reproducibility"],
    )


@flyte.trace
async def _write_report(lit_review: LiteratureReview, plan: ResearchPlan) -> ResearchReport:
    """Generate a research report. Replaces ProfessorAgent's report writing phase."""
    text = await _llm(
        user=(
            f"Topic: {lit_review.topic}\n"
            f"Hypothesis: {plan.hypothesis}\n"
            f"Methodology: {plan.methodology}\n\n"
            "Write a short research report:\n"
            "TITLE:\n...\n\nABSTRACT:\n...\n\nMETHODOLOGY:\n...\n\nRESULTS:\n...\n\nCONCLUSION:\n..."
        ),
        max_tokens=1024,
    )
    sections = {"TITLE": "", "ABSTRACT": "", "METHODOLOGY": "", "RESULTS": "", "CONCLUSION": ""}
    current = None
    for line in text.splitlines():
        for key in sections:
            if line.strip().upper().startswith(key + ":") or line.strip().upper() == key:
                current = key
                remainder = line.split(":", 1)[1].strip() if ":" in line else ""
                if remainder:
                    sections[current] += remainder + " "
                break
        else:
            if current and line.strip():
                sections[current] += line.strip() + " "
    return ResearchReport(
        title=sections["TITLE"].strip() or f"Research on {lit_review.topic}",
        abstract=sections["ABSTRACT"].strip(),
        methodology_description=sections["METHODOLOGY"].strip(),
        results=sections["RESULTS"].strip(),
        conclusion=sections["CONCLUSION"].strip(),
    )

### 6. Define the ReviewersAgent

The original `ReviewersAgent.inference()` called `get_score()` three times sequentially, each with a different reviewer persona. In Flyte v2, `asyncio.gather` runs all three reviewers concurrently — same tripartite judgment mechanism, 3× faster.

The reviewer personas are preserved verbatim from the original.

In [ ]:
REVIEWER_PERSONAS = [
    # From original ReviewersAgent.inference()
    "You are a harsh but fair reviewer who expects good experiments that lead to insights for the research topic.",
    "You are a harsh and critical but fair reviewer who is looking for an idea that would be impactful in the field.",
    "You are a harsh but fair open-minded reviewer that is looking for novel ideas that have not been proposed before.",
]

REVIEW_SYSTEM = """\
Review the research paper against the provided plan. Return ONLY valid JSON (no markdown fences).
Be concise: keep "summary" to 1-2 sentences and each of "strengths"/"weaknesses" to at most 3 short items.
{
  "summary": "<brief summary>",
  "strengths": ["<strength>"],
  "weaknesses": ["<weakness>"],
  "originality": <1-4>,
  "quality": <1-4>,
  "clarity": <1-4>,
  "significance": <1-4>,
  "soundness": <1-4>,
  "overall": <1-10>,
  "decision": "Accept" | "Reject",
  "ethical_concerns": false
}"""


@flyte.trace
async def _single_review(
    plan: ResearchPlan,
    report: ResearchReport,
    reviewer_persona: str,
) -> ReviewScore:
    """One reviewer's evaluation. Replaces get_score() in the original."""
    import json
    client = AsyncAnthropic(max_retries=5)  # backoff for transient HTTP 500s
    response = await client.messages.create(
        # max_tokens must comfortably exceed the JSON review or the response is truncated
        # mid-string -> json.loads raises "Unterminated string". 512 was too tight once a
        # reviewer wrote longer strengths/weaknesses; 1500 leaves headroom.
        model="claude-haiku-4-5-20251001",
        max_tokens=1500,
        system=f"{reviewer_persona}\n\n{REVIEW_SYSTEM}",
        messages=[{
            "role": "user",
            "content": (
                f"Research plan:\nHypothesis: {plan.hypothesis}\nMethod: {plan.methodology}\n\n"
                f"Report title: {report.title}\nAbstract: {report.abstract}\n"
                f"Results: {report.results}\nConclusion: {report.conclusion}"
            ),
        }],
    )

    if response.stop_reason == "max_tokens":
        raise RuntimeError(
            "Reviewer response was truncated at max_tokens; raise max_tokens or shorten the rubric."
        )

    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return ReviewScore.model_validate(json.loads(raw))


@flyte.trace
async def _reviewers_agent(plan: ResearchPlan, report: ResearchReport) -> ReviewPanel:
    """
    Tripartite review — three concurrent reviewers.

    Replaces ReviewersAgent.inference() which called get_score() sequentially:
      review_1 = get_score(..., reviewer_type=reviewer_1, ...)
      review_2 = get_score(..., reviewer_type=reviewer_2, ...)
      review_3 = get_score(..., reviewer_type=reviewer_3, ...)

    asyncio.gather runs all three in parallel — same judgment, 3x faster.
    """
    import asyncio
    r1, r2, r3 = await asyncio.gather(
        _single_review(plan, report, REVIEWER_PERSONAS[0]),
        _single_review(plan, report, REVIEWER_PERSONAS[1]),
        _single_review(plan, report, REVIEWER_PERSONAS[2]),
    )
    consensus = (r1.overall + r2.overall + r3.overall) / 3
    accepts = sum(1 for r in [r1, r2, r3] if r.decision == "Accept")
    decision = "Accept" if accepts >= 2 else "Reject"
    return ReviewPanel(
        reviewer_1=r1, reviewer_2=r2, reviewer_3=r3,
        consensus_score=consensus,
        final_decision=decision,
    )

### 7. Define the research orchestrator task

The original Agent Laboratory sequenced phases via class method calls (`postdoc.context()`, `professor.generate_readme()`). In Flyte v2, these are explicit `@flyte.trace` calls inside a single `@env.task` — the same DAG, visible as checkpoints in the UI.

In [ ]:
def _html_escape(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


@discovery_env.task(
    retries=1,
    timeout=timedelta(minutes=20),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def research_pipeline(
    research_topic: str,
) -> ResearchResult:
    """
    Full research pipeline: literature → plan → report → review.

    Replaces Agent Laboratory's multi-class orchestration:
      phd_agent.run() → postdoc.context("plan formulation") →
      postdoc.context("results interpretation") → professor.generate_readme() →
      reviewers.inference(plan, report)

    @flyte.trace on each phase provides checkpointing — the task resumes from the
    last successful phase on pod failure.
    """
    # Phase 1: Literature review (literature review agent)
    lit_review = await _literature_agent(topic=research_topic)
    await flyte.report.replace.aio(
        f"<h2>{_html_escape(research_topic)}</h2>"
        f"<h3>Phase 1: Literature Review</h3>"
        f"<p>{_html_escape(lit_review.summary)}</p>"
        f"<h4>Key Findings</h4><ul>" +
        "".join(f"<li>{_html_escape(f)}</li>" for f in lit_review.key_findings) +
        "</ul><h4>Research Gaps</h4><ul>" +
        "".join(f"<li>{_html_escape(g)}</li>" for g in lit_review.research_gaps) +
        "</ul>"
    )
    await flyte.report.flush.aio()

    # Phase 2: Plan formulation (PostdocAgent)
    plan = await _postdoc_plan(lit_review=lit_review)
    await flyte.report.log.aio(
        f"<h3>Phase 2: Research Plan</h3>"
        f"<p><strong>Hypothesis:</strong> {_html_escape(plan.hypothesis)}</p>"
        f"<p><strong>Method:</strong> {_html_escape(plan.methodology)}</p>"
    )
    await flyte.report.flush.aio()

    # Phase 3: Report writing (ProfessorAgent)
    report = await _write_report(lit_review=lit_review, plan=plan)
    await flyte.report.log.aio(
        f"<h3>Phase 3: Research Report — {_html_escape(report.title)}</h3>"
        f"<p><em>{_html_escape(report.abstract)}</em></p>"
    )
    await flyte.report.flush.aio()

    # Phase 4: Tripartite peer review (ReviewersAgent — 3 concurrent reviewers)
    panel = await _reviewers_agent(plan=plan, report=report)
    decision_color = "green" if panel.final_decision == "Accept" else "red"
    await flyte.report.log.aio(
        f"<h3>Phase 4: Peer Review</h3>"
        f"<p>Consensus score: <strong>{panel.consensus_score:.1f}/10</strong> | "
        f"Decision: <span style='color:{decision_color}'><strong>{panel.final_decision}</strong></span></p>"
        f"<table border='1' cellpadding='4'>"
        f"<tr><th>Reviewer</th><th>Overall</th><th>Originality</th><th>Decision</th></tr>"
        f"<tr><td>Harsh/fair</td><td>{panel.reviewer_1.overall}</td><td>{panel.reviewer_1.originality}</td><td>{panel.reviewer_1.decision}</td></tr>"
        f"<tr><td>Critical/impact</td><td>{panel.reviewer_2.overall}</td><td>{panel.reviewer_2.originality}</td><td>{panel.reviewer_2.decision}</td></tr>"
        f"<tr><td>Open-minded/novel</td><td>{panel.reviewer_3.overall}</td><td>{panel.reviewer_3.originality}</td><td>{panel.reviewer_3.decision}</td></tr>"
        f"</table>"
    )
    await flyte.report.flush.aio()

    # Generate README (ProfessorAgent.generate_readme() equivalent)
    client = AsyncAnthropic(max_retries=5)  # backoff for transient HTTP 500s
    readme_response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        system="You are a professor writing a GitHub README for a research project. Be concise.",
        messages=[{
            "role": "user",
            "content": f"Paper: {report.title}\nAbstract: {report.abstract}\nConclusion: {report.conclusion}\n\nWrite a markdown README:",
        }],
    )
    readme = readme_response.content[0].text.strip()

    return ResearchResult(
        topic=research_topic,
        literature_review=lit_review,
        research_plan=plan,
        report=report,
        review_panel=panel,
        readme=readme,
    )

### 8. Run locally

In [10]:
run = flyte.run(
    research_pipeline,
    research_topic="Efficient fine-tuning of large language models for domain-specific tasks with limited labeled data",
)
run.wait()
result: ResearchResult = run.outputs()[0]

print(f"Paper: {result.report.title}")
print(f"Abstract: {result.report.abstract[:200]}...")
print(f"\nReview panel decision: {result.review_panel.final_decision}")
print(f"Consensus score: {result.review_panel.consensus_score:.1f}/10")
print(f"\nLiterature gaps found: {len(result.literature_review.research_gaps)}")
print(f"\nREADME preview:\n{result.readme[:400]}...")

> Building 1 image...

> Building image discovery-agent for environment discovery_agent

✓ Built image for environment discovery_agent: localhost:30000/discovery-agent:6e5ffb6b434f24a301b355b5aff6ad29

Output()

Paper: Research on Efficient fine-tuning of large language models for domain-specific tasks with limited labeled data
Abstract: ...

Review panel decision: Reject
Consensus score: 3.0/10

Literature gaps found: 11

README preview:
# Efficient Fine-tuning of LLMs for Domain-Specific Tasks

## Overview

This repository contains research and implementation code for efficient fine-tuning of large language models (LLMs) on domain-specific tasks with limited labeled data. We explore parameter-efficient approaches to adapt pre-trained LLMs without requiring full model retraining.

## Key Contributions

- **Parameter-Efficient Meth...


## Why not the Agent harness here?

This pattern's value is in showing the **bare search/sampling loop** with explicit, deterministic bookkeeping. An agent *can* drive exploration — the **learning-and-adaptation** notebook (#9) does exactly that with `CodeModeAgent` and `flyte.sandbox` — but here, hiding the loop inside the harness would obscure the very mechanism the notebook is meant to teach.